# M2.1 · k-NN ve mesafe matrisi vektörizasyonu

Vektörizasyonun klasik okulu. Dört noktada mesafeleri ve oyları elle hesaplıyor; mesafe matrisini önce iki döngüyle, sonra tek döngüyle, sonra hiç döngüsüz (‖a‖² + ‖b‖² − 2a·b açılımı) yazıyor; PyTorch'ta `topk` ile sınıflandırıyor ve en sonda 20 farklı k değerini tek sıralamayla değerlendiriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-1-knn.html](https://m-gonul.github.io/pytorch-lab/m2-1-knn.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_1_knn.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref_pred(D, y, k):\n    out = []\n    for row in D:\n        order = sorted(range(len(row)), key=lambda j: (row[j], j))[:k]\n        cnt = {}\n        for j in order:\n            cnt[y[j]] = cnt.get(y[j], 0) + 1\n        out.append(min(cnt, key=lambda c: (-cnt[c], c)))\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    Xtr, ytr = [[0, 0], [1, 3], [3, 1], [4, 4]], [0, 1, 0, 1]\n    R.test(\"elle örnek: [[5, 1, 5, 13]]\", lambda: close(need(ns, \"sqdist_loops\")([[1, 2]], Xtr), [[5, 1, 5, 13]]))\n\n    def hand_pred():\n        D = [[5, 1, 5, 13]]\n        ensure(list(need(ns, \"knn_predict\")(D, ytr, 1)) == [1], \"k=1 için [1] bekleniyordu\")\n        ensure(list(need(ns, \"knn_predict\")(D, ytr, 3)) == [0], \"k=3 için [0] bekleniyordu\")\n    R.test(\"elle örnek: k=1 → 1, k=3 → 0\", hand_pred)\n\n    def rnd():\n        rng = random.Random(0)\n        A = [[rng.uniform(-2, 2) for _ in range(3)] for _ in range(6)]\n        B = [[rng.uniform(-2, 2) for _ in range(3)] for _ in range(9)]\n        close(need(ns, \"sqdist_loops\")(A, B), [[sum((a - b) ** 2 for a, b in zip(x, z)) for z in B] for x in A], what=\"6×9 mesafe\")\n    R.test(\"rastgele 6×9 mesafe matrisi\", rnd)\n\n    def ties():\n        D = [[1, 1, 2, 2], [3, 1, 1, 0]]\n        y = [1, 0, 0, 1]\n        for k in (1, 2, 4):\n            got = list(need(ns, \"knn_predict\")(D, y, k))\n            ensure(got == ref_pred(D, y, k), f\"k={k}: {got} bulundu, {ref_pred(D, y, k)} bekleniyordu (eşitlik kuralları)\")\n    R.test(\"eşitlik kuralları (mesafe ve oy)\", ties)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef ref_pred(D, y, k):\n    out = []\n    for row in D:\n        order = sorted(range(len(row)), key=lambda j: (row[j], j))[:k]\n        cnt = {}\n        for j in order:\n            cnt[int(y[j])] = cnt.get(int(y[j]), 0) + 1\n        out.append(min(cnt, key=lambda c: (-cnt[c], c)))\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"sqdist_one_loop: en fazla tek döngü\", lambda: max_loops(src, 1, func=\"sqdist_one_loop\"))\n    R.test(\"sqdist_no_loop: döngü yok\", lambda: max_loops(src, 0, func=\"sqdist_no_loop\"))\n    R.test(\"knn_predict_np: döngü yok\", lambda: max_loops(src, 0, func=\"knn_predict_np\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(1)\n    A, B = rng.normal(size=(7, 5)), rng.normal(size=(11, 5))\n    ref = ((A[:, None] - B[None]) ** 2).sum(-1)\n    R.test(\"sqdist_one_loop doğru\", lambda: close(need(ns, \"sqdist_one_loop\")(A, B), ref))\n    R.test(\"sqdist_no_loop doğru ve ≥ 0\", lambda: (close(need(ns, \"sqdist_no_loop\")(A, B), ref, rtol=1e-9, atol=1e-9),\n                                                 ensure(np.all(need(ns, \"sqdist_no_loop\")(A, A) >= 0), \"negatif mesafe var; np.maximum(D, 0) eksik\")))\n\n    def pred():\n        f = need(ns, \"knn_predict_np\")\n        ytr = rng.integers(0, 4, size=11)\n        for k in (1, 3, 5, 11):\n            got = np.asarray(f(ref, ytr, k)).tolist()\n            ensure(got == ref_pred(ref, ytr, k), f\"k={k}: {got} ≠ {ref_pred(ref, ytr, k)}\")\n        D = np.array([[1, 1, 2, 2], [3, 1, 1, 0]], dtype=float)\n        y = np.array([1, 0, 0, 1])\n        for k in (1, 2, 4):\n            ensure(np.asarray(f(D, y, k)).tolist() == ref_pred(D, y, k), f\"eşitlik durumu k={k}\")\n    R.test(\"knn_predict_np: rastgele + eşitlik kuralları\", pred)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure, max_loops\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: (max_loops(src, 0, func=\"sqdist_torch\"), max_loops(src, 0, func=\"knn_predict_torch\")))\n    g = torch.Generator().manual_seed(2)\n    A, B = torch.randn(30, 8, generator=g, dtype=torch.float64), torch.randn(50, 8, generator=g, dtype=torch.float64)\n    R.test(\"sqdist_torch = cdist²\", lambda: close(need(ns, \"sqdist_torch\")(A, B), torch.cdist(A, B) ** 2, rtol=1e-9, atol=1e-9))\n    R.test(\"sqdist_torch(A, A) ≥ 0\", lambda: ensure(bool((need(ns, \"sqdist_torch\")(A.float(), A.float()) >= 0).all()), \"negatif değer var; clamp_min(0) eksik\"))\n\n    def acc():\n        centers = torch.tensor([[0.0, 3.0], [-3.0, -2.0], [3.0, -2.0]])\n        ytr = torch.arange(3).repeat_interleave(80)\n        Xtr = centers[ytr] + torch.randn(240, 2, generator=g)\n        yte = torch.arange(3).repeat_interleave(30)\n        Xte = centers[yte] + torch.randn(90, 2, generator=g)\n        p = need(ns, \"knn_predict_torch\")(Xte, Xtr, ytr, 7, 3)\n        ensure(tuple(p.shape) == (90,), f\"shape {tuple(p.shape)}\")\n        a = (p == yte).float().mean().item()\n        ensure(a >= 0.85, f\"test doğruluğu {a:.2f}\")\n        D = torch.cdist(Xte, Xtr) ** 2\n        idx = D.argsort(1)[:, :7]\n        ref = torch.nn.functional.one_hot(ytr[idx], 3).sum(1).argmax(1)\n        ensure(bool((p == ref).all()), \"referans k-NN ile farklı tahminler\")\n        return f\"3 sınıf, k=7, test doğruluğu {a:.3f}\"\n    R.test(\"knn_predict_torch: referansla aynı, doğruluk ≥ 0.85\", acc)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import no_loops, need, ensure, bench, fmt_time\n\n\ndef one_k(D, ytr, k, C):\n    idx = torch.sort(D, dim=1, stable=True).indices[:, :k]\n    return F.one_hot(ytr[idx], C).sum(1).argmax(1)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"elle örnek: k = 1, 2, 3, 4 → 1, 0, 0, 0\",\n           lambda: ensure(need(ns, \"predict_many_k\")(torch.tensor([[5.0, 1.0, 5.0, 13.0]]), torch.tensor([0, 1, 0, 1]), torch.tensor([1, 2, 3, 4]), 2).reshape(-1).tolist() == [1, 0, 0, 0],\n                          \"k=2'de oylar 1–1 berabere: küçük etiket (0) kazanmalı\"))\n    g = torch.Generator().manual_seed(4)\n    Xtr, Xte = torch.randn(400, 6, generator=g), torch.randn(120, 6, generator=g)\n    ytr = torch.randint(0, 5, (400,), generator=g)\n    D = torch.cdist(Xte, Xtr) ** 2\n    ks = torch.arange(1, 21)\n\n    def eq():\n        P = need(ns, \"predict_many_k\")(D, ytr, ks, 5)\n        ensure(tuple(P.shape) == (20, 120), f\"shape {tuple(P.shape)}, (20, 120) bekleniyordu\")\n        for i, k in enumerate(ks.tolist()):\n            ensure(bool((P[i] == one_k(D, ytr, k, 5)).all()), f\"k={k} tahminleri farklı\")\n        return \"20 k değerinin hepsi ayrı ayrı hesaplananla aynı\"\n    R.test(\"20 k değeri, tek tek tahminle eşleşiyor\", eq)\n    if \"predict_many_k\" not in ns:\n        return\n    Xtr2, Xte2 = torch.randn(5000, 64, generator=g), torch.randn(500, 64, generator=g)\n    ytr2 = torch.randint(0, 10, (5000,), generator=g)\n    D2 = torch.cdist(Xte2, Xtr2) ** 2\n    t_loop = bench(lambda: [one_k(D2, ytr2, k, 10) for k in range(1, 21)], repeat=3)\n    t_vec = bench(lambda: ns[\"predict_many_k\"](D2, ytr2, ks, 10), repeat=3)\n    R.table(\"20 k değeri (500 test × 5000 eğitim)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for k: her seferinde sırala + oyla\", fmt_time(t_loop), \"1×\"],\n             [\"senin predict_many_k (tek sıralama + cumsum)\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n    te, tr = Xte2[:200], Xtr2\n    t_bc = bench(lambda: ((te[:, None] - tr[None]) ** 2).sum(-1), repeat=3)\n    t_ex = bench(lambda: ((te ** 2).sum(1, keepdim=True) + (tr ** 2).sum(1) - 2 * te @ tr.T).clamp_min(0), repeat=3)\n    t_cd = bench(lambda: torch.cdist(te, tr), repeat=3)\n    R.table(\"Mesafe matrisi, 200 × 5000, d = 64\", [\"yöntem\", \"süre\", \"ara bellek\"],\n            [[\"broadcasting fark: (200, 5000, 64)\", fmt_time(t_bc), f\"{200 * 5000 * 64 * 4 / 2**20:,.0f} MB\"],\n             [\"açılım: ‖a‖² + ‖b‖² − 2ab\", fmt_time(t_ex), f\"{200 * 5000 * 4 / 2**20:,.1f} MB\"],\n             [\"torch.cdist\", fmt_time(t_cd), \"kütüphaneye bağlı\"]],\n            note=\"Açılım, işi tek bir matris çarpımına (BLAS) taşıdığı için hem hızlı hem bellek dostu.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m2_1_knn as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

k-en yakın komşu öğrenmez, ezberler: tahmin zamanında sorgu noktasına en yakın $k$ eğitim örneğini bulur ve çoğunluk oyuna bakar. Algoritmanın tüm maliyeti bir mesafe matrisinde:

$$
D_{ij}=\lVert x^{\text{test}}_i-x^{\text{train}}_j\rVert^2,\qquad D\in\mathbb{R}^{n_{te}\times n_{tr}}.
$$

Bu modül CS231n'in ilk ödevinin izinden gidiyor: aynı $D$'yi iki döngüyle, tek döngüyle ve hiç döngüsüz yazmak. Döngüsüz hâlin anahtarı kareyi açmak:

$$
\lVert a-b\rVert^2=\lVert a\rVert^2+\lVert b\rVert^2-2\,a\cdot b
\quad\Longrightarrow\quad
D = \underbrace{s_{te}}_{(n_{te},1)} + \underbrace{s_{tr}^\top}_{(1,n_{tr})} - 2\,X_{te}X_{tr}^\top .
$$

Son terim tek bir matris çarpımı (M0.2); ilk ikisi broadcasting ile $(n_{te}\times n_{tr})$'ye yayılır.

**Eşitlik kuralları** (bütün aşamalarda aynı): mesafesi eşit komşulardan indeksi küçük olan önce gelir; oylar eşitse küçük sınıf etiketi kazanır.

## 1 · Elle

Dört noktayı ve sorguyu kareli kâğıda işaretle. Kare mesafeleri hesapla (karekök gerekmez: sıralamayı değiştirmez).

$k=1$ ile $k=3$'ün farklı karar verdiğine dikkat et. $k$ küçükse model tek bir (belki gürültülü) komşuya güvenir; büyükse sınırları yumuşatır ama küçük sınıfları ezebilir. $k$, bir learning rate gibi doğrulama setinde seçilen bir hiperparametredir; (e) aşamasında bunu verimli yapacaksın.

Son grupta, $d_4^2$'yi açılım formülüyle yeniden hesapla. Aynı sayının çıkması (c) aşamasındaki döngüsüz yöntemin doğruluğunun kanıtı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Eğitim: $P_1=(0,0)$ sınıf 0, $P_2=(1,3)$ sınıf 1, $P_3=(3,1)$ sınıf 0, $P_4=(4,4)$ sınıf 1. Sorgu $q=(1,2)$. Kare mesafeler $\lVert q-P_i\rVert^2$**

- `d1`: $d_1^2$
- `d2`: $d_2^2$
- `d3`: $d_3^2$
- `d4`: $d_4^2$

**Oylama**

- `k1`: $k=1$ tahmini (sınıf)
- `k3`: $k=3$ tahmini (sınıf)

**Açılım: $\lVert q-P_4\rVert^2=\lVert q\rVert^2+\lVert P_4\rVert^2-2\,q\cdot P_4$**

- `qq`: $\lVert q\rVert^2$
- `pp`: $\lVert P_4\rVert^2$
- `qp`: $q\cdot P_4$
- `dd`: $5+32-2\cdot12$

In [ ]:
cevaplar = dict(
    d1=None,
    d2=None,
    d3=None,
    d4=None,
    k1=None,
    k3=None,
    qq=None,
    pp=None,
    qp=None,
    dd=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"d1":[5,0],"d2":[1,0],"d3":[5,0],"d4":[13,0],"k1":[1,0],"k3":[0,0],"qq":[5,0],"pp":[32,0],"qp":[12,0],"dd":[13,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Listelerle iki fonksiyon:

- `sqdist_loops(Xte, Xtr)` — iki iç içe döngüyle `nte × ntr` kare mesafe listesi.
- `knn_predict(D, ytr, k)` — her test satırı için: komşuları `(mesafe, indeks)` sırasına göre diz, ilk `k`'nın etiketlerini say (`dict` ya da `collections.Counter`), en çok oyu alanı seç; eşitlikte küçük etiket.

`numpy`/`torch` yasak.

In [ ]:
%%writefile stage_b.py
def sqdist_loops(Xte, Xtr):
    """→ nte × ntr kare mesafe listesi (iki döngü)."""
    ...


def knn_predict(D, ytr, k):
    """D: nte × ntr. → tahmin listesi. Eşit mesafe: küçük indeks önce; eşit oy: küçük etiket."""
    ...


Xtr = [[0, 0], [1, 3], [3, 1], [4, 4]]
ytr = [0, 1, 0, 1]
D = sqdist_loops([[1, 2]], Xtr)
print(D)                                     # [[5, 1, 5, 13]]
print(knn_predict(D, ytr, 1), knn_predict(D, ytr, 3))   # [1] [0]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Üç sürüm yaz; kontrol her birinin döngü derinliğini ayrı ayrı denetler:

- `sqdist_one_loop(Xte, Xtr)` — test satırları üzerinde **tek** döngü; her satır için `((Xtr - x) ** 2).sum(1)` (broadcasting).
- `sqdist_no_loop(Xte, Xtr)` — **döngü yok**; açılım formülü. Negatif çıkan küçük değerleri `np.maximum(D, 0)` ile kırp.
- `knn_predict_np(D, ytr, k)` — **döngü yok**. Sıralama için `np.argsort(D, axis=1, kind="stable")` (eşit mesafede küçük indeks önce gelir). Oylar: komşu etiketlerini one-hot'a çevirip topla — `np.eye(C)[ytr[idx]]` shape'i `(n_te, k, C)`, `sum(axis=1)` sonrası `argmax` (eşitlikte ilk, yani küçük etiket).

In [ ]:
%%writefile stage_c.py
import numpy as np


def sqdist_one_loop(Xte, Xtr):
    ...


def sqdist_no_loop(Xte, Xtr):
    ...


def knn_predict_np(D, ytr, k):
    ...


Xtr = np.array([[0, 0], [1, 3], [3, 1], [4, 4]], dtype=float)
ytr = np.array([0, 1, 0, 1])
D = sqdist_no_loop(np.array([[1.0, 2.0]]), Xtr)
print(D, knn_predict_np(D, ytr, 1), knn_predict_np(D, ytr, 3))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta aynı açılım; ek olarak hazır `torch.cdist` var. Sıralamanın tamamı gerekmez, yalnız en küçük $k$: `torch.topk(D, k, largest=False)` $O(n)$ seçim yapar, tam sıralama $O(n\log n)$.

- `sqdist_torch(Xte, Xtr)` — açılım ile, `clamp_min(0)`.
- `knn_predict_torch(Xte, Xtr, ytr, k, C)` — `sqdist_torch` + `topk` + `F.one_hot(...).sum(1).argmax(1)`.

Kontrol, `sqdist_torch`'u `torch.cdist(...) ** 2` ile karşılaştırır ve 3 sınıflı bir veri setinde doğruluğa bakar. Gradyan yok: k-NN'in öğrenilen parametresi yok.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def sqdist_torch(Xte, Xtr):
    ...


def knn_predict_torch(Xte, Xtr, ytr, k, C):
    ...


torch.manual_seed(0)
Xtr, Xte = torch.randn(100, 2), torch.randn(5, 2)
ytr = (Xtr[:, 0] > 0).long()
print(knn_predict_torch(Xte, Xtr, ytr, 5, 2), (Xte[:, 0] > 0).long())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

En iyi $k$'yı seçmek için birçok $k$ dener. Her $k$ için yeniden sıralayıp oy saymak gereksiz: sıralama bir kez yapılır, oylar **kümülatif** toplanır.

$$
V_{i,m,c}=\sum_{r=1}^{m}[\,y_{\text{nb}(i,r)}=c\,]\quad\Rightarrow\quad \hat y_i^{(k)}=\arg\max_c V_{i,k,c}.
$$

`one_hot` ile `(n_te, k_max, C)` tensörü kur, `cumsum(dim=1)` al; $k$ için gereken satır `V[:, k-1, :]`. Tüm $k$'lar tek indekslemeyle: `V[:, ks - 1, :]`.

- `predict_many_k(D, ytr, ks, C)` → `(len(ks), n_te)`. **Döngü yok.** Sıralama için `torch.sort(D, dim=1, stable=True)` kullan (b–c'deki eşitlik kuralı).

Kontrol, sonucu her $k$ için ayrı tahminle karşılaştırır ve 20 $k$ değeri için hız tablosu üretir; ayrıca mesafe hesabının üç yolunu yarıştırır.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def predict_many_k(D, ytr, ks, C):
    """D: (nte, ntr), ks: (m,) uzun tensör → (m, nte) tahminler. Döngü yok."""
    ...


D = torch.tensor([[5.0, 1.0, 5.0, 13.0]])
ytr = torch.tensor([0, 1, 0, 1])
print(predict_many_k(D, ytr, torch.tensor([1, 2, 3, 4]), 2))   # [[1],[0],[0],[0]]

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Cover & Hart (1967): en yakın komşu sınıflandırması

[Cover & Hart — Nearest Neighbor Pattern Classification (IEEE Trans. Information Theory 1967)](https://ieeexplore.ieee.org/document/1053964)

Makalenin ana sonucunu deneyle doğrula: Bayes hatası R* bilinen sentetik problemlerde (iki Gauss, örtüşme ayarlı) 1-NN'in asimptotik hatasının R* ≤ R ≤ 2R*(1 − R*) aralığında kaldığını, örnek sayısı n = 10²…10⁵ için göster. k-NN'de k ve n birlikte büyürken (k → ∞, k/n → 0) hatanın R*'a yaklaştığını çiz; boyut arttıkça yakınsamanın yavaşladığını (boyut laneti) ölç.


**Adımlar**

1. R*'ı iki Gauss için kapalı form ya da büyük Monte Carlo ile hesapla.

**Kabul ölçütleri**

- Ölçülen 1-NN hataları Cover–Hart sınırları içinde.
- k-NN hatasının R*'a yaklaşma grafiği ve boyut etkisi.

## Kaynaklar

- [CS231n — Image Classification: Data-driven Approach, k-Nearest Neighbor](https://cs231n.github.io/classification/) — MIT
- [CS231n Assignment 1 — kNN (compute_distances_two_loops / one_loop / no_loops)](https://cs231n.github.io/assignments2025/assignment1/) — MIT
- [ISLP — Bölüm 2.2.3 K-Nearest Neighbors](https://www.statlearning.com/) — doğrulanmadı
- [PyTorch — torch.cdist, torch.topk](https://docs.pytorch.org/docs/stable/generated/torch.cdist.html) — BSD-3